# 02 – Baseline Model
### AMR Prediction | DRIAMS-A 2018 – Ciprofloxacin

Logistic Regression ile ilk baseline modeli eğitiyoruz.  
5-Fold Stratified CV ile AUROC, AUPRC, F1 değerlerini ölçüyoruz.

## 1. Veriyi Yükle

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

from src.data.load_driams import load_driams

print("Veri yükleniyor...")
X, y, meta = load_driams("DRIAMS-A", "2018", "Ciprofloxacin")

print(f"Toplam örnek : {len(y):,}")
print(f"R (dirençli) : {y.sum()} ({y.mean()*100:.1f}%)")
print(f"S (duyarlı)  : {(1-y).sum()} ({(1-y.mean())*100:.1f}%)")
print(f"X shape      : {X.shape}")

Veri yükleniyor...
Toplam örnek : 8,094
R (dirençli) : 1644 (20.3%)
S (duyarlı)  : 6450 (79.7%)
X shape      : (8094, 6000)


## 2. Pipeline Oluştur

Preprocessing: `StandardScaler`  
Model: `LogisticRegressionCV` — `Cs=[0.001, 0.01, 0.1, 1, 10]` üzerinde iç CV ile en iyi C'yi otomatik seçer.  
`saga` + L1: 6000 boyutlu MALDI verisi için hızlı ve sparse çözüm.

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV

from src.features.preprocessing import build_preprocessing_pipeline

preprocess = build_preprocessing_pipeline(scale=True, use_pca=False)

# Outer CV için fixed-C: nested CV + double n_jobs problemini önler
lr_cv = LogisticRegression(
    C=0.1, solver="saga", penalty="l1",
    max_iter=5000, random_state=42, class_weight="balanced",
)

pipeline = Pipeline([
    *preprocess.steps,
    ("classifier", lr_cv),
])

print(pipeline)

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(C=0.1, class_weight='balanced',
                                    max_iter=5000, penalty='l1',
                                    random_state=42, solver='saga'))])


## 3. 5-Fold Stratified Cross-Validation

In [ ]:
from src.models.train import train_cv

cv_results = train_cv(pipeline, X, y, n_splits=5)

## 4. Sonuçları Kaydet

In [ ]:
RESULTS_DIR = Path("../outputs/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# CV fold detaylarını kaydet
cv_results.to_csv(RESULTS_DIR / "lr_ciprofloxacin_cv_folds.csv", index=False)

# Özet satırı (ortalama ± std)
metrics = ["roc_auc", "average_precision", "f1_macro", "accuracy"]
summary_rows = []
for m in metrics:
    col = f"test_{m}"
    summary_rows.append({
        "model": "LogisticRegression",
        "antibiotic": "Ciprofloxacin",
        "dataset": "DRIAMS-A 2018",
        "metric": m,
        "mean": round(cv_results[col].mean(), 4),
        "std": round(cv_results[col].std(), 4),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS_DIR / "lr_ciprofloxacin_summary.csv", index=False)

print("Sonuçlar kaydedildi:")
print(f"  {RESULTS_DIR}/lr_ciprofloxacin_cv_folds.csv")
print(f"  {RESULTS_DIR}/lr_ciprofloxacin_summary.csv")
print()
print(summary_df.to_string(index=False))

## 5. ROC & PR Eğrileri – Son Fold Üzerinde

In [ ]:
from sklearn.model_selection import StratifiedKFold
from src.evaluation.metrics import evaluate, plot_roc_pr, plot_confusion_matrix

# Son fold'u yeniden çalıştır (görselleştirme için)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(X, y))
train_idx, test_idx = folds[-1]

pipeline.fit(X[train_idx], y[train_idx])

best_C = pipeline.named_steps["classifier"].C_[0]
print(f"Seçilen C (iç CV): {best_C}")

y_prob = pipeline.predict_proba(X[test_idx])[:, 1]

result = evaluate(y[test_idx], y_prob, label="LR – Ciprofloxacin (fold 5)")
y_pred_opt = (y_prob >= result["threshold"]).astype(int)

In [ ]:
plot_roc_pr(y[test_idx], y_prob, label="LR_Ciprofloxacin")
plot_confusion_matrix(y[test_idx], y_pred_opt, label="LR_Ciprofloxacin")

## 6. Modeli Kaydet (Full Dataset)

In [ ]:
from src.models.train import save_model

# Son model: tüm veri üzerinde LogisticRegressionCV ile en iyi C'yi bul
final_pipeline = Pipeline([
    *preprocess.steps,
    ("classifier", LogisticRegressionCV(
        Cs=[0.001, 0.01, 0.1, 1.0, 10.0],
        solver="saga", penalty="l1", cv=5,
        max_iter=5000, random_state=42, class_weight="balanced", n_jobs=-1,
    )),
])
final_pipeline.fit(X, y)
best_C = final_pipeline.named_steps["classifier"].C_[0]
print(f"Seçilen C (full data): {best_C}")

save_model(final_pipeline, "lr_ciprofloxacin_baseline")
print("Baseline model kaydedildi.")